# Notebook 6 — ChangeTX Text-to-Change Search from a Hosted Retrieval Bundle

### *Hands-On Session: Earth Embeddings for EO — Retrieval, Discovery, and Change-Oriented Search*

**Instructors:** Fuxun Yu, Rishi Madhok (TerraByte AI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iceysteel/ESA-NASA-Workshop-2026/blob/main/Day%202/Track%202/Earth-Embeddings-EO/notebook_6_changetx_text_to_change_search.ipynb)

---

## What we're doing

Notebook 5 introduced **text-to-change retrieval** with a simple `normalize(t2 - t1)` embedding. Here we use the actual ChangeTX bi-temporal model behind the demo.

The hosted bundle contains clean before/after images, cached retrieval embeddings, tokenizer / image-processor files, and the full fine-tuned SigLIP + ChangeTX adapter weights. The Python model class is **not** stored on the Hub; it lives in the workshop repo as `changetx_adapter.py`, and this notebook imports it.

Pipeline:

1. Download the artifact-only bundle from Hugging Face.
2. Import the local ChangeTX adapter class.
3. Load the full model with `ChangeTXBiTemporalModel.from_pretrained(...)`.
4. Use cached embeddings for fast search.
5. Optionally regenerate embeddings from the clean pre/post images to show the full path.

## Runtime

**CPU is fine for search.** The first run downloads about **1.2 GB** because the bundle now includes the full model plus clean pre/post images. After that, Hugging Face cache reuse makes startup much faster.

Interactive query time is cheap: one text forward pass and a `1398 x 768` dot product. Regenerating all image-pair embeddings is the expensive offline step; the notebook only regenerates a tiny subset for demonstration.

In [ ]:
!pip install -q huggingface_hub transformers sentencepiece protobuf matplotlib pillow

In [ ]:
import json
import os
import textwrap
import time
import urllib.request
import zipfile
from pathlib import Path

os.environ.setdefault('HF_HUB_DISABLE_XET', '1')

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from huggingface_hub import snapshot_download
from PIL import Image
from transformers import SiglipTokenizer
try:
    from transformers import SiglipImageProcessorPil as SiglipImageProcessor
except ImportError:
    from transformers import SiglipImageProcessor

# Keep this notebook CPU-stable for workshop machines and Colab CPU runtimes.
device = 'cpu'
torch.set_grad_enabled(False)
print('Using device:', device)

## 1. Import the repo-local adapter class

The Hub repo stores artifacts only. If this notebook is opened directly in Colab, the cell below pulls `changetx_adapter.py` from the workshop GitHub repo. If you cloned the repo locally, it imports the file already next to the notebook.

In [ ]:
ADAPTER_FILE = Path('changetx_adapter.py')
if not ADAPTER_FILE.exists():
    ADAPTER_URL = (
        'https://raw.githubusercontent.com/iceysteel/ESA-NASA-Workshop-2026/main/'
        'Day%202/Track%202/Earth-Embeddings-EO/changetx_adapter.py'
    )
    urllib.request.urlretrieve(ADAPTER_URL, ADAPTER_FILE)
    print('Downloaded adapter from GitHub')

from changetx_adapter import ChangeTXBiTemporalConfig, ChangeTXBiTemporalModel

## 2. Download the hosted ChangeTX bundle

The Hub bundle stores model weights, tokenizer files, image processor config, cached features, metadata, and `images.zip`. We download the small file set, then unzip the clean images locally so the notebook avoids thousands of individual Hub file requests.

In [ ]:
HF_BUNDLE_REPO = 'FuxunTB/changetx-text-change-demo'

t0 = time.time()
bundle_dir = Path(snapshot_download(repo_id=HF_BUNDLE_REPO, repo_type='dataset', max_workers=2))
manifest = json.loads((bundle_dir / 'manifest.json').read_text())
print(f'Downloaded/resolved in {time.time()-t0:.1f}s')
print('Bundle dir:', bundle_dir)
print('Samples:', manifest['sample_count'])
print('Embedding dim:', manifest['embedding_dim'])
print('Bundle version:', manifest['bundle_version'])

image_root = Path('changetx_text_change_demo_images')
images_info = manifest['images']
archive_path = bundle_dir / images_info.get('archive_path', 'images.zip')
expected_pre_dir = image_root / images_info.get('pre_dir', 'images/pre')
expected_post_dir = image_root / images_info.get('post_dir', 'images/post')

if not expected_pre_dir.exists() or not expected_post_dir.exists():
    t_zip = time.time()
    image_root.mkdir(parents=True, exist_ok=True)
    root_resolved = image_root.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (image_root / member.filename).resolve()
            if root_resolved != target and root_resolved not in target.parents:
                raise RuntimeError(f'Unsafe zip member: {member.filename}')
        archive.extractall(image_root)
    print(f'Unzipped clean images in {time.time()-t_zip:.1f}s to {image_root}')
else:
    print('Using already-unzipped clean images:', image_root)

## 3. Load cached embeddings and sample metadata

`image_features` is the fast searchable index: one normalized fused embedding per before/after pair. The same bundle also contains clean images, so we can reproduce these embeddings with the full model when needed.

In [ ]:
blob = np.load(bundle_dir / manifest['features']['path'])
image_features = F.normalize(torch.from_numpy(blob['image_features']).float(), dim=-1).cpu()
caption_text_features = F.normalize(torch.from_numpy(blob['caption_text_features']).float(), dim=-1).cpu()

samples = [json.loads(line) for line in (bundle_dir / manifest['samples']['path']).read_text().splitlines()]
N, D = image_features.shape

assert len(samples) == N
assert D == manifest['embedding_dim']
print(f'image_features: {tuple(image_features.shape)}  dtype={image_features.dtype}')
print(f'caption_text_features: {tuple(caption_text_features.shape)}')
print('Example sample:', {k: samples[1][k] for k in ['pair_key', 'caption_change', 'pre_image_path', 'post_image_path']})

## 4. Browse random clean before/after pairs

These are the actual pre/post images extracted from `images.zip`, not pre-rendered panels. The notebook assembles the comparison view on the fly.

In [ ]:
def load_pair(index: int):
    sample = samples[int(index)]
    pre = Image.open(image_root / sample['pre_image_path']).convert('RGB')
    post = Image.open(image_root / sample['post_image_path']).convert('RGB')
    return pre, post

def show_pair_rows(indices, scores=None, title=None):
    rows = len(indices)
    fig, axes = plt.subplots(rows, 2, figsize=(7.5, 2.6 * rows))
    axes = np.asarray(axes).reshape(rows, 2)
    for row, idx in enumerate(indices):
        sample = samples[int(idx)]
        pre, post = load_pair(int(idx))
        prefix = '' if scores is None else f'cos={scores[row]:.3f} | '
        caption = textwrap.shorten(sample['caption_change'], width=78)
        axes[row, 0].imshow(pre)
        axes[row, 0].set_title('pre', fontsize=9)
        axes[row, 0].set_ylabel(prefix + caption, fontsize=9)
        axes[row, 0].axis('off')
        axes[row, 1].imshow(post)
        axes[row, 1].set_title('post', fontsize=9)
        axes[row, 1].axis('off')
    if title:
        plt.suptitle(title, y=1.005, fontsize=11)
    plt.tight_layout()
    plt.show()

def show_random_pairs(n: int = 5, seed: int = 7):
    rng = np.random.default_rng(seed)
    picks = rng.choice(N, size=min(n, N), replace=False)
    show_pair_rows(picks, title='Random ChangeTX pairs')

show_random_pairs(n=10, seed=96)

## 5. Load the full ChangeTX model

The bundle has the full SigLIP vision tower, SigLIP text tower, temporal position vectors, and fusion adapter. `changetx_adapter.py` defines the small wrapper class; all weights are loaded from the artifact bundle.

In [ ]:
t0 = time.time()
model_config = ChangeTXBiTemporalConfig.from_pretrained(str(bundle_dir), local_files_only=True)
model = ChangeTXBiTemporalModel.from_pretrained(str(bundle_dir), config=model_config, local_files_only=True).eval().to(device)
tokenizer = SiglipTokenizer.from_pretrained(str(bundle_dir), local_files_only=True)
image_processor = SiglipImageProcessor.from_pretrained(str(bundle_dir), local_files_only=True)
max_length = int(getattr(tokenizer, 'model_max_length', 64) or 64)
if max_length > 10000:
    max_length = 64

print(f'Model loaded in {time.time()-t0:.1f}s')
print('Model class:', type(model).__name__)
print('Tokenizer max_length:', max_length)

## 6. Reproduce a few cached embeddings from clean images

This is the same operation the offline generation script performs over the full pool. We only run a few pairs here to keep the tutorial interactive on CPU.

In [ ]:
def encode_texts(texts):
    encoded = tokenizer(
        list(texts),
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    ).to(device)
    with torch.inference_mode():
        return model.encode_text(
            encoded['input_ids'],
            attention_mask=encoded.get('attention_mask'),
            normalize=True,
        ).cpu()

def encode_pairs(indices):
    pre_images, post_images = [], []
    for idx in indices:
        pre, post = load_pair(int(idx))
        pre_images.append(pre)
        post_images.append(post)
    pre = image_processor(images=pre_images, return_tensors='pt')['pixel_values'].to(device)
    post = image_processor(images=post_images, return_tensors='pt')['pixel_values'].to(device)
    with torch.inference_mode():
        return model.encode_pair(pre, post, normalize=True).cpu()

check_indices = [0, 1, 2, 3]
regen_image = encode_pairs(check_indices)
regen_text = encode_texts([samples[i]['caption_change'] for i in check_indices])
print('image cosines vs cached:', torch.sum(regen_image * image_features[check_indices], dim=-1).numpy().round(5))
print('text cosines vs cached:', torch.sum(regen_text * caption_text_features[check_indices], dim=-1).numpy().round(5))

## 7. Text-to-change search

Interactive search uses the cached image-pair embeddings. The text query is embedded with the same full model loaded above.

In [ ]:
def encode_text(query: str) -> torch.Tensor:
    return encode_texts([query])[0]

def search_change(query: str, k: int = 6):
    t0 = time.time()
    qvec = encode_text(query)
    enc_ms = (time.time() - t0) * 1000
    sims = image_features @ qvec
    top = torch.topk(sims, k=min(k, N)).indices.cpu().numpy()
    return top, sims[top].cpu().numpy(), enc_ms

def show_results(query: str, k: int = 6):
    top, scores, ms = search_change(query, k=k)
    show_pair_rows(top, scores=scores, title=f'Query: "{query}"   (text encode: {ms:.0f} ms)')
    return top, scores

## 8. Demo queries

ChangeTX contains construction, parking/storage changes, vegetation/seasonal shifts, roads, water/soil appearance, and many subtle no-change examples.

In [ ]:
show_results('new construction', k=6)

In [ ]:
show_results('parking lot constructed', k=6)

In [ ]:
show_results('nothing changed in the scene', k=6)

In [ ]:
show_results('new solar panel', k=6)

## 9. Try your own query

Edit the string below and re-run. Useful prompts to try:

- `new houses were constructed`
- `construction site established`
- `vegetation was cleared for construction`
- `parking lot constructed`
- `storage facility established`
- `seasonal vegetation change`
- `nothing changed`

In [ ]:
show_results('storage facility established', k=6)